<a href="https://colab.research.google.com/github/dhiyasalmas/Parallel-Data-Processing/blob/main/Belaja%20paralel.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **MPI in Colab**

In [ ]:
# Menginstal OpenMPI di sistem Linux Ubuntu milik Colab
!sudo apt-get install openmpi-bin libopenmpi-dev

# Menginstal library Python untuk MPI
!pip install mpi4py

Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
libopenmpi-dev is already the newest version (4.1.2-2ubuntu1).
libopenmpi-dev set to manually installed.
openmpi-bin is already the newest version (4.1.2-2ubuntu1).
openmpi-bin set to manually installed.
0 upgraded, 0 newly installed, 0 to remove and 37 not upgraded.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 64.5 MB/s eta 0:00:00


In [ ]:
%%writefile simpson.py
from mpi4py import MPI
import timeit
import math
comm = MPI.COMM_WORLD
rank = comm.Get_rank()
size = comm.Get_size()
start = MPI.Wtime()

def y(x):
    return (4/(1+x**2))

def simpson(a, b, n):
    h = (b - a) / n
    s = y(a) + y(b)

    for i in range(1, n):
        x = a + i * h
        if i % 2 == 0:
            s += 2 * y(x)  # untuk indeks genap
        else:
            s += 4 * y(x)  # untuk indeks ganjil

    return (h / 3) * s

a = 0
b = 1
n = 10**8
h = (b - a)/n

local_n = n // size
local_a = a + rank * local_n * h
local_b = local_a + local_n * h

local_integral = simpson(local_a, local_b, local_n)

my_integral=0
if rank == 0:
    my_integral = local_integral
    for i in range(1,size):
        my_integral_2 = comm.recv(source=MPI.ANY_SOURCE)
        my_integral = my_integral + my_integral_2
    print("hasil numerik = ", my_integral)
    print("waktu = ", MPI.Wtime()-start, "detik")
else:
    comm.send(local_integral,dest=0)

Overwriting simpson.py


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 python simpson.py

hasil numerik =  3.1415926535899263
waktu =  29.747214824 detik


# **CUDA in Colab**

In [ ]:
!nvidia-smi

Tue Feb 24 12:05:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   52C    P8             14W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2025 NVIDIA Corporation
Built on Fri_Feb_21_20:23:50_PST_2025
Cuda compilation tools, release 12.8, V12.8.93
Build cuda_12.8.r12.8/compiler.35583870_0


## Simpson 1/3

$$\int_a^b f(x) dx \approx \frac{h}{3} \left[ f(x_0) + 4 \sum_{i=1,3,5}^{n-1} f(x_i) + 2 \sum_{i=2,4,6}^{n-2} f(x_i) + f(x_n) \right]$$

In [ ]:
import numpy as np
import cupy as cp
import time

# ==========================================
# 1. Definisi Fungsi
# ==========================================
# Fungsi untuk dieksekusi oleh CPU
def f_cpu(x):
    return 4.0 / (1.0 + x**2)

# Fungsi untuk dieksekusi oleh GPU
def f_gpu(x):
    return 4.0 / (1.0 + x**2)

# ==========================================
# 2. Logika Integrasi Simpson 1/3
# ==========================================
def simpson_cpu(a, b, n):
    if n % 2 != 0: n += 1
    h = (b - a) / n
    x = np.linspace(a, b, n + 1)
    y = f_cpu(x)
    return (h / 3.0) * (y[0] + 4 * np.sum(y[1:-1:2]) + 2 * np.sum(y[2:-2:2]) + y[-1])

def simpson_gpu(a, b, n):
    if n % 2 != 0: n += 1
    h = (b - a) / n
    x = cp.linspace(a, b, n + 1)
    y = f_gpu(x)
    return (h / 3.0) * (y[0] + 4 * cp.sum(y[1:-1:2]) + 2 * cp.sum(y[2:-2:2]) + y[-1])

# ==========================================
# 3. Parameter Simulasi
# ==========================================
a = 0.0          # Batas bawah
b = 1.0          # Batas atas
n = 10**8  # 100 Juta pias (beban kerja yang sangat berat)

print(f"Menghitung integral 4/(1+x^2) dari {a} ke {b}")
print(f"Jumlah pias: {n:,}\n")

# ==========================================
# 4. Eksekusi di CPU (NumPy)
# ==========================================
print("Mulai menghitung di CPU...")
start_cpu = time.time()
hasil_cpu = simpson_cpu(a, b, n)
waktu_cpu = time.time() - start_cpu

print(f"Hasil CPU  : {hasil_cpu:.10f}")
print(f"Waktu CPU  : {waktu_cpu:.4f} detik\n")

# ==========================================
# 5. Eksekusi di GPU (CuPy)
# ==========================================
# Pemanasan GPU (Compile kernel CUDA pertama kali)
_ = simpson_gpu(0, 1, 10)

print("Mulai menghitung di GPU T4...")
start_gpu = time.time()
hasil_gpu = simpson_gpu(a, b, n)
cp.cuda.Stream.null.synchronize() # Wajib: Sinkronisasi agar timer akurat
waktu_gpu = time.time() - start_gpu

print(f"Hasil GPU  : {hasil_gpu:.10f}")
print(f"Waktu GPU  : {waktu_gpu:.4f} detik\n")

# ==========================================
# 6. Kesimpulan
# ==========================================
speedup = waktu_cpu / waktu_gpu
print("-" * 30)
print(f"KESIMPULAN:")
print(f"GPU {speedup:.1f}x LEBIH CEPAT dibandingkan CPU")

Menghitung integral 4/(1+x^2) dari 0.0 ke 1.0
Jumlah pias: 100,000,000

Mulai menghitung di CPU...
Hasil CPU  : 3.1415926536
Waktu CPU  : 1.0713 detik

Mulai menghitung di GPU T4...
Hasil GPU  : 3.1415926536
Waktu GPU  : 0.3150 detik

------------------------------
KESIMPULAN:
GPU 3.4x LEBIH CEPAT dibandingkan CPU


In [ ]:
import numpy as np
from numba import cuda

# ==========================================
# 1. Mendefinisikan Kernel CUDA
# ==========================================
@cuda.jit
def tambah_vektor_gpu(A, B, C):
    # cuda.grid(1) memberikan ID unik untuk setiap thread (pekerja).
    # Karena ada ribuan thread yang jalan bareng, tiap thread
    # perlu tahu dia kebagian tugas menghitung elemen indeks ke berapa.
    i = cuda.grid(1)

    # Pastikan ID pekerja tidak melebihi jumlah data yang ada
    if i < A.size:
        C[i] = A[i] + B[i]  # Pekerja ke-i menjumlahkan data ke-i

# ==========================================
# 2. Persiapan Data di CPU (Host)
# ==========================================
N = 1_000_000  # 1 Juta elemen
array_A = np.ones(N)      # Array berisi angka 1
array_B = np.ones(N) * 2  # Array berisi angka 2
array_C = np.zeros(N)     # Array kosong untuk menampung hasil

# ==========================================
# 3. Konfigurasi Pasukan GPU (Blocks & Threads)
# ==========================================
# GPU butuh instruksi: berapa banyak pekerja yang dikerahkan?
threads_per_block = 256
# Menghitung jumlah grup (block) yang dibutuhkan agar semua 1 juta data tercover
blocks_per_grid = (N + (threads_per_block - 1)) // threads_per_block

print(f"Mengerahkan {blocks_per_grid} grup, di mana tiap grup berisi {threads_per_block} pekerja.")

# ==========================================
# 4. Eksekusi Kernel di GPU
# ==========================================
# Sintaks peluncuran kernel Numba: nama_fungsi[grid, block](argumen)
tambah_vektor_gpu[blocks_per_grid, threads_per_block](array_A, array_B, array_C)

# Tampilkan 5 hasil pertama untuk memverifikasi
print("Hasil 5 elemen pertama:", array_C)

Mengerahkan 3907 grup, di mana tiap grup berisi 256 pekerja.
Hasil 5 elemen pertama: [3. 3. 3. ... 3. 3. 3.]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/cudadrv/devicearray.py:937: NumbaPerformanceWarning: Host array used in CUDA kernel will incur copy overhead to/from device.
  warn(NumbaPerformanceWarning(msg))


In [ ]:
from numba import cuda, types
import numpy as np

@cuda.jit
def stencil_1d_shared(d_in, d_out):
    # 1. Alokasi Papan Tulis (Shared Memory)
    # Kita pesankan array sebesar 256 elemen bertipe float32 di dalam chip SM
    s_data = cuda.shared.array(shape=256, dtype=types.float32)

    # Dapatkan nomor urut thread di dalam ruangan (Block) ini: 0 sampai 255
    tx = cuda.threadIdx.x
    # Dapatkan ID global thread ini di seluruh pasukan
    i = cuda.grid(1)

    if i < d_in.size:
        # 2. KERJA BAKTI: Setiap thread mengambil 1 data dari Global Memory
        # lalu menaruhnya di slot papan tulis miliknya masing-masing
        s_data[tx] = d_in[i]

    # 3. SINKRONISASI! (Hukum Besi Shared Memory)
    # Kita WAJIB menyuruh semua thread berhenti sejenak di sini sampai
    # proses kerja bakti memindahkan data selesai 100%.
    # Jika tidak, thread yang bekerja sangat cepat mungkin akan mencoba membaca
    # data tetangganya padahal tetangganya belum selesai menaruh data di papan.
    cuda.syncthreads()

    # 4. Fase Komputasi: Menggunakan Shared Memory yang super cepat
    if i < d_in.size:
        # Pengecualian batas agar tidak error saat membaca array[tx-1] atau array[tx+1]
        if tx > 0 and tx < 255:
            # Hitung rata-rata perpindahan dengan mengambil data murni dari papan tulis!
            d_out[i] = (s_data[tx - 1] + s_data[tx] + s_data[tx + 1]) / 3.0
        else:
            d_out[i] = d_in[i] # Batas ujung tetap sama

# **CUDA C++**

In [ ]:
%%writefile vektor.cu
#include <iostream>
#include <cuda_runtime.h>

__global__ void tambah_vektor(float *A, float *B, float *C, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

int main() {
    int N = 1000000;
    size_t ukuran_bytes = N * sizeof(float);

    float *h_A = (float*)malloc(ukuran_bytes);
    float *h_B = (float*)malloc(ukuran_bytes);
    float *h_C = (float*)malloc(ukuran_bytes);

    for (int i = 0; i < N; i++) {
        h_A[i] = 1.0f;
        h_B[i] = 2.0f;
    }

    float *d_A, *d_B, *d_C;
    cudaMalloc(&d_A, ukuran_bytes);
    cudaMalloc(&d_B, ukuran_bytes);
    cudaMalloc(&d_C, ukuran_bytes);

    cudaMemcpy(d_A, h_A, ukuran_bytes, cudaMemcpyHostToDevice);
    cudaMemcpy(d_B, h_B, ukuran_bytes, cudaMemcpyHostToDevice);

    int threads_per_block = 256;
    int blocks_per_grid = (N + threads_per_block - 1) / threads_per_block;

    tambah_vektor<<<blocks_per_grid, threads_per_block>>>(d_A, d_B, d_C, N);

    cudaMemcpy(h_C, d_C, ukuran_bytes, cudaMemcpyDeviceToHost);

    std::cout << "Komputasi di GPU T4 Selesai!" << std::endl;
    std::cout << "Hasil indeks ke-0 (1.0 + 2.0) = " << h_C[0] << std::endl;
    std::cout << "Hasil indeks ke-999999 (1.0 + 2.0) = " << h_C[999999] << std::endl;

    cudaFree(d_A); cudaFree(d_B); cudaFree(d_C);
    free(h_A); free(h_B); free(h_C);

    return 0;
}

Writing vektor.cu


In [ ]:
!nvcc vektor.cu -o program_vektor

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!./program_vektor

Komputasi di GPU T4 Selesai!
Hasil indeks ke-0 (1.0 + 2.0) = 3
Hasil indeks ke-999999 (1.0 + 2.0) = 3


In [ ]:
# Jalankan ini sekali di awal
!pip install nvcc4jupyter
%load_ext nvcc4jupyter

Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpy3nl8a4n".


In [ ]:
%%cuda
#include <iostream>
#include <cuda_runtime.h>

// ==========================================
// 1. KERNEL GPU
// ==========================================
__global__ void kuadratkan_array(float *d_out, float *d_in, int N) {
    // Menghitung ID pekerja secara mandiri
    int i = blockIdx.x * blockDim.x + threadIdx.x;

    // Setiap pekerja mengkuadratkan elemen bagiannya sendiri
    if (i < N) {
        d_out[i] = d_in[i] * d_in[i];
    }
}

// ==========================================
// 2. FUNGSI UTAMA CPU
// ==========================================
int main() {
    int N = 5; // Kita pakai 5 angka saja agar mudah dilihat hasilnya
    size_t ukuran = N * sizeof(float);

    // a. Data awal di CPU (Host)
    float h_in[5] = {1.0f, 2.0f, 3.0f, 4.0f, 5.0f};
    float h_out[5];

    // b. Alokasi memori kosong di GPU (Device)
    float *d_in, *d_out;
    cudaMalloc(&d_in, ukuran);
    cudaMalloc(&d_out, ukuran);

    // c. Copy data awal dari CPU ke GPU
    cudaMemcpy(d_in, h_in, ukuran, cudaMemcpyHostToDevice);

    // d. Eksekusi Kernel (1 Block berisi 256 Threads)
    int threads_per_block = 256;
    int blocks_per_grid = (N + threads_per_block - 1) / threads_per_block;

    kuadratkan_array<<<blocks_per_grid, threads_per_block>>>(d_out, d_in, N);

    // e. Copy hasil dari GPU kembali ke CPU
    cudaMemcpy(h_out, d_out, ukuran, cudaMemcpyDeviceToHost);

    // f. Tampilkan Hasil
    std::cout << "Data Awal di CPU   : ";
    for(int i = 0; i < N; i++) std::cout << h_in[i] << " ";
    std::cout << "\n";

    std::cout << "Hasil Kuadrat GPU  : ";
    for(int i = 0; i < N; i++) std::cout << h_out[i] << " ";
    std::cout << "\n";

    // g. Bersihkan memori GPU
    cudaFree(d_in);
    cudaFree(d_out);

    return 0;
}

Data Awal di CPU   : 1 2 3 4 5 
Hasil Kuadrat GPU  : 1 4 9 16 25 



In [ ]:
%%cuda
#include <iostream>
#include <cuda_runtime.h>

// ==========================================
// 1. KERNEL GPU (Sama persis seperti Level 3)
// ==========================================
__global__ void tambah_vektor(float *A, float *B, float *C, int N) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < N) {
        C[i] = A[i] + B[i];
    }
}

// ==========================================
// 2. FUNGSI UTAMA CPU
// ==========================================
int main() {
    int N = 1000000; // 1 Juta data partikel
    size_t ukuran = N * sizeof(float);

    // a. Deklarasi SATU pointer saja untuk CPU & GPU
    float *A, *B, *C;

    // b. Keajaiban Unified Memory: cudaMallocManaged
    cudaMallocManaged(&A, ukuran);
    cudaMallocManaged(&B, ukuran);
    cudaMallocManaged(&C, ukuran);

    // c. Inisialisasi data langsung di CPU (Tidak perlu malloc dan cudaMemcpy!)
    for (int i = 0; i < N; i++) {
        A[i] = 1.0f;
        B[i] = 2.0f;
    }

    // d. Konfigurasi dan Eksekusi Kernel GPU
    int threads_per_block = 256;
    int blocks_per_grid = (N + threads_per_block - 1) / threads_per_block;

    // GPU langsung memproses pointer A, B, C yang sama
    tambah_vektor<<<blocks_per_grid, threads_per_block>>>(A, B, C, N);

    // e. WAJIB: Sinkronisasi!
    // Kita harus menyuruh CPU berhenti sejenak dan menunggu GPU selesai bekerja
    // sebelum CPU mencoba membaca hasilnya.
    cudaDeviceSynchronize();

    // f. CPU langsung membaca hasilnya dari pointer C
    std::cout << "Berhasil menggunakan Unified Memory!" << std::endl;
    std::cout << "Hasil indeks ke-0      (1.0 + 2.0) = " << C[0] << std::endl;
    std::cout << "Hasil indeks ke-999999 (1.0 + 2.0) = " << C[N-1] << std::endl;

    // g. Bersihkan memori
    cudaFree(A);
    cudaFree(B);
    cudaFree(C);

    return 0;
}

Berhasil menggunakan Unified Memory!
Hasil indeks ke-0      (1.0 + 2.0) = 3
Hasil indeks ke-999999 (1.0 + 2.0) = 3



# **MPI, CUDA**

In [ ]:
!sudo apt-get update
!sudo apt-get install -y openmpi-bin libopenmpi-dev

Get:1 https://cli.github.com/packages stable InRelease [3,917 B]
Get:2 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [357 B]
Hit:5 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:6 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ Packages [85.0 kB]
Get:7 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,383 kB]
Get:8 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:9 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:10 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB]
Get:11 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubuntu jammy-

In [ ]:
%%writefile hibrida.cu
#include <mpi.h>
#include <cuda_runtime.h>
#include <iostream>

__global__ void komputasi_berat_gpu(float *data, int ukuran) {
    int i = blockIdx.x * blockDim.x + threadIdx.x;
    if (i < ukuran) {
        data[i] = data[i] * 2.0f;
    }
}

int main(int argc, char** argv) {
    MPI_Init(&argc, &argv);

    int rank, size;
    MPI_Comm_rank(MPI_COMM_WORLD, &rank);
    MPI_Comm_size(MPI_COMM_WORLD, &size);

    int jumlah_gpu;
    cudaGetDeviceCount(&jumlah_gpu);
    int gpu_id = rank % jumlah_gpu;
    cudaSetDevice(gpu_id);

    int beban_per_rank = 10; // Kita kecilkan agar mudah dicetak
    float *data_lokal;
    cudaMallocManaged(&data_lokal, beban_per_rank * sizeof(float));

    for(int i=0; i<beban_per_rank; i++) data_lokal[i] = 1.0f;

    int threads = 256;
    int blocks = (beban_per_rank + threads - 1) / threads;
    komputasi_berat_gpu<<<blocks, threads>>>(data_lokal, beban_per_rank);

    cudaDeviceSynchronize();

    float total_lokal = 0.0f;
    for(int i=0; i<beban_per_rank; i++) {
        total_lokal += data_lokal[i];
    }

    float total_global = 0.0f;
    MPI_Reduce(&total_lokal, &total_global, 1, MPI_FLOAT, MPI_SUM, 0, MPI_COMM_WORLD);

    if (rank == 0) {
        std::cout << "Total global dari " << size << " proses MPI: " << total_global << std::endl;
    }

    cudaFree(data_lokal);
    MPI_Finalize();
    return 0;
}

Writing hibrida.cu


In [ ]:
!nvcc hibrida.cu -o program_hibrida -I/usr/lib/x86_64-linux-gnu/openmpi/include/openmpi -I/usr/lib/x86_64-linux-gnu/openmpi/include -L/usr/lib/x86_64-linux-gnu/openmpi/lib -lmpi

nvcc warning : Support for offline compilation for architectures prior to '<compute/sm/lto>_75' will be removed in a future release (Use -Wno-deprecated-gpu-targets to suppress warning).


In [ ]:
!mpirun --allow-run-as-root --oversubscribe -n 2 ./program_hibrida

Total global dari 2 proses MPI: 40
